# Notebook 01: Exposure class and population served (Google Earth Engine)
**Workstreams:** inputs for 1B, 1C, 1D

**What this notebook does**
1. Reads ESA WorldCover 10 m in Earth Engine and converts land cover to surface roughness z₀ around every tower (500 m radius) within 100 km of the coast
2. From z₀, assigns each tower
   - a **WMO/TD-1555 exposure class** (in-land, off-land or off-sea), which sets the gust factor in Notebooks 02–05
   - an **IS 875 / IS 802 terrain category** (1–4), used in the code comparison (Notebook 04)
3. Builds a service area (Voronoi polygon) for each substation and sums WorldPop 2020 population inside it, giving the **population served** used for outage costs

If Earth Engine is not available, the notebook falls back to a distance-to-coast rule for exposure and to Raj et al.'s population weights, so later notebooks still run.

**Needs:** Notebook 00 run first, and the GEE project `goggle-earth-499404` registered for non-commercial use.

**Assumption to note in the write-up:** exposure is set from roughness around the tower, not from the upwind direction of each storm. That is a simplification; Notebook 04 shows how much it matters.

In [ ]:
# ==== Shared functions (cdri_lib): identical copy in every notebook from 01 onwards ====
# Change a function here, rebuild the notebooks, and every notebook uses the same version.
import os, sys, re, json, math, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm
from scipy.optimize import minimize
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
pd.set_option('display.width', 160)

KT2MS, NMI2KM, R_EARTH_KM = 0.514444, 1.852, 6371.0
SURF_TO_GRADIENT = 0.85          # surface-to-gradient wind ratio used only to estimate the Holland B parameter
CRS_GEO, CRS_M = 'EPSG:4326', 'EPSG:32645'

# WMO/TD-1555 (Harper, Kepert & Ginger 2010) Table 1.1: 3-s gust factor G(3, To) at +10 m in tropical cyclones
GUST3 = {'in-land':  {60: 1.49, 180: 1.58, 600: 1.66},
         'off-land': {60: 1.36, 180: 1.44, 600: 1.52},
         'off-sea':  {60: 1.23, 180: 1.31, 600: 1.38},
         'at-sea':   {60: 1.11, 180: 1.17, 600: 1.23}}


def setup_project():
    """Mount Google Drive (in Colab) and return the project root and folder dictionary."""
    if 'google.colab' in sys.modules:
        from google.colab import drive
        try:
            drive.mount('/content/drive')
            root = Path('/content/drive/MyDrive/CDRI_Odisha')
        except Exception as e:
            print('Drive mount failed:', e, '\nFix: Runtime → Disconnect and delete runtime, rerun, use the same Google '
                  'account and tick all permission boxes. Using temporary storage for now.')
            root = Path('/content/CDRI_Odisha')
    else:
        root = Path(os.environ.get('CDRI_ROOT', Path.cwd() / 'CDRI_Odisha'))
    dirs = {k: root / v for k, v in {'raw': 'data/raw', 'external': 'data/external', 'interim': 'data/interim',
                                     'processed': 'data/processed', 'figures': 'outputs/figures',
                                     'tables': 'outputs/tables'}.items()}
    for d in dirs.values():
        d.mkdir(parents=True, exist_ok=True)
    print('Project root:', root)
    return root, dirs


def union(gs):
    """Dissolve a GeoSeries/GeoDataFrame to one geometry (old and new geopandas)."""
    return gs.union_all() if hasattr(gs, 'union_all') else gs.unary_union


def need(path, notebook):
    """Stop with a clear message if an input from an earlier notebook is missing."""
    if not Path(path).exists():
        raise FileNotFoundError(f'{path} not found. Run Notebook {notebook} first.')
    return Path(path)


def gust_factor(exposure, avg_s):
    """WMO 3-s gust factor for each exposure class, given the averaging period (s) of the mean wind."""
    return np.array([GUST3[e][int(avg_s)] for e in np.asarray(exposure, dtype=object)], dtype=float)


def haversine_km(lat1, lon1, lat2, lon2):
    la1, lo1, la2, lo2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = np.sin((la2 - la1) / 2) ** 2 + np.cos(la1) * np.cos(la2) * np.sin((lo2 - lo1) / 2) ** 2
    return 2 * R_EARTH_KM * np.arcsin(np.sqrt(np.clip(a, 0, 1)))


def _interp_valid(tn, t, y):
    y = np.asarray(y, float)
    ok = np.isfinite(y)
    if ok.sum() == 0:
        return np.full(len(tn), np.nan)
    if ok.sum() == 1:
        return np.full(len(tn), y[ok][0])
    return np.interp(tn, t[ok], y[ok])


def interp_track(t_h, lat, lon, vmax, rmw, pc=None, dt_h=1.0):
    """Interpolate a track to dt_h-hour steps and add the translation velocity (u east, v north; m/s)."""
    t_h = np.asarray(t_h, float)
    if len(t_h) < 2:
        return None
    tn = np.arange(t_h[0], t_h[-1] + 1e-9, dt_h)
    tr = {'t_h': tn, 'lat': _interp_valid(tn, t_h, lat), 'lon': _interp_valid(tn, t_h, lon),
          'vmax': _interp_valid(tn, t_h, vmax), 'rmw': _interp_valid(tn, t_h, rmw),
          'pc': _interp_valid(tn, t_h, pc) if pc is not None else np.full(len(tn), np.nan)}
    if len(tn) < 2:
        tr['u'] = tr['v'] = np.zeros(len(tn))
        return tr
    dy = np.gradient(tr['lat']) * 111.195
    dx = np.gradient(tr['lon']) * 111.195 * np.cos(np.radians(tr['lat']))
    dt = np.gradient(tn) * 3600.0
    tr['u'], tr['v'] = dx * 1000 / dt, dy * 1000 / dt
    return tr


def holland_max_wind(tr, plat, plon, p_env=1008.0, b_fixed=None, b_default=1.6, rho=1.15,
                     max_dist_km=600.0, chunk=4000):
    """Storm-maximum surface wind at points (same averaging basis as tr['vmax']).

    Holland (1980) radial profile scaled to the symmetric maximum wind, plus a translation asymmetry
    (Jelesnianski 1966: Vt·Rm·r/(Rm²+r²), half the translation speed at the RMW, strongest right of track).
    B comes from the central-pressure deficit (Holland 1980) when pressure is known, else b_default; clipped to 1.0–2.5.
    No land-roughness reduction is applied (as in STORM / Bloemendaal 2020); exposure is handled by the gust factor.
    """
    plat, plon = np.asarray(plat, float), np.asarray(plon, float)
    out = np.zeros(len(plat))
    lat, lon = tr['lat'][:, None], tr['lon'][:, None]
    u, v = tr['u'][:, None], tr['v'][:, None]
    vm, rm = tr['vmax'][:, None], np.maximum(tr['rmw'][:, None], 5.0)
    vt = np.hypot(u, v)
    vsym = np.maximum(vm - 0.5 * vt, 0.5 * vm)
    if b_fixed is not None:
        B = np.full_like(vm, float(b_fixed))
    else:
        dp = (p_env - tr['pc'][:, None]) * 100.0
        vg = vsym / SURF_TO_GRADIENT
        B = np.where(np.isfinite(dp) & (dp > 200.0), rho * np.e * vg ** 2 / np.maximum(dp, 200.0), b_default)
        B = np.clip(B, 1.0, 2.5)
    coslat = np.cos(np.radians(lat))
    for s in range(0, len(plat), chunk):
        pl, po = plat[None, s:s + chunk], plon[None, s:s + chunk]
        dy = (pl - lat) * 111.195
        dx = (po - lon) * 111.195 * coslat
        r = np.maximum(np.hypot(dx, dy), 0.5)
        x = (rm / r) ** B
        vs = vsym * np.sqrt(x * np.exp(1.0 - x))
        proj = (-dy * u + dx * v) / r                     # translation component along the cyclonic tangential direction
        w = vs + proj * (rm * r / (rm ** 2 + r ** 2))
        w = np.where(r <= max_dist_km, w, 0.0)
        out[s:s + chunk] = np.maximum(w.max(axis=0), 0.0)
    return out


def frag_p(im, theta, beta):
    """Lognormal fragility: P(damage | IM)."""
    im = np.maximum(np.asarray(im, float), 1e-3)
    return norm.cdf((np.log(im) - np.log(theta)) / beta)


def fit_fragility(im, y, theta0=None, beta0=0.2):
    """Maximum-likelihood lognormal fragility from binary outcomes (Baker 2015). Returns theta, beta, -loglik."""
    im, y = np.asarray(im, float), np.asarray(y, bool)
    ok = np.isfinite(im) & (im > 0)
    im, y = im[ok], y[ok]
    if theta0 is None:
        theta0 = np.median(im[y]) if y.any() else np.percentile(im, 99)
    li = np.log(im)

    def nll(p):
        pr = np.clip(norm.cdf((li - p[0]) / np.exp(p[1])), 1e-12, 1 - 1e-12)
        return -(y * np.log(pr) + (~y) * np.log(1 - pr)).sum()
    res = minimize(nll, [np.log(theta0), np.log(beta0)], method='Nelder-Mead',
                   options={'xatol': 1e-7, 'fatol': 1e-7, 'maxiter': 5000})
    return float(np.exp(res.x[0])), float(np.exp(res.x[1])), float(res.fun)


def bootstrap_fragility(im, y, n=200, seed=1):
    rng = np.random.default_rng(seed)
    im, y = np.asarray(im, float), np.asarray(y, bool)
    th, be = [], []
    for _ in range(n):
        i = rng.integers(0, len(im), len(im))
        if y[i].sum() < 3:
            continue
        t, b, _ = fit_fragility(im[i], y[i])
        th.append(t); be.append(b)
    return np.array(th), np.array(be)


# ---------- project data loaders ----------
def load_towers(dirs):
    """Towers from Notebook 00, with exposure from Notebook 01 if available (else a distance-to-coast rule)."""
    t = pd.read_parquet(need(dirs['processed'] / 'towers.parquet', '00'))
    ex = dirs['processed'] / 'towers_exposure.parquet'
    if ex.exists():
        e = pd.read_parquet(ex)
        t = t.merge(e[['tower_id', 'z0', 'terrain_cat', 'exposure', 'exposure_src']], on='tower_id', how='left')
    else:
        t['z0'], t['terrain_cat'], t['exposure'], t['exposure_src'] = np.nan, np.nan, None, None
    miss = t.exposure.isna()
    t.loc[miss, 'exposure'] = np.where(t.loc[miss, 'dist_coast_km'] <= 2.0, 'off-sea', 'in-land')
    t.loc[miss, 'exposure_src'] = 'rule: ≤2 km from coast = off-sea, else in-land'
    t['terrain_cat'] = t.terrain_cat.fillna(2).astype(int)
    return t


def load_network(dirs):
    """Lines and substations (with population served) from Notebooks 00/01."""
    import geopandas as gpd
    gpkg = need(dirs['processed'] / 'odisha_network.gpkg', '00')
    lines = gpd.read_file(gpkg, layer='lines')
    subs = gpd.read_file(gpkg, layer='substations')
    popf = dirs['processed'] / 'substations_population.csv'
    if popf.exists():
        p = pd.read_csv(popf)
        subs = subs.merge(p[['sub_id', 'pop_worldpop']], on='sub_id', how='left')
        subs['pop'] = subs.pop_worldpop.fillna(subs.pop_weight)
        subs['pop_src'] = np.where(subs.pop_worldpop.notna(), 'WorldPop 2020 service area', 'Raj et al. weight')
    else:
        subs['pop'], subs['pop_src'] = subs.pop_weight, 'Raj et al. weight'
    subs['pop'] = subs['pop'].fillna(0.0)
    return lines, subs


def load_fragility(dirs, which='any_damage'):
    f = dirs['processed'] / 'fragility_params.json'
    if f.exists():
        d = json.loads(f.read_text())
        if which in d:
            return d[which]['theta'], d[which]['beta'], d[which].get('source', 'Notebook 02')
    print('  ! fragility_params.json not found: using Raj et al. collapse fragility (292.5 km/h, β = 0.104)')
    return 292.5 / 3.6, 0.104, 'Raj et al. (collapse)'


# ---------- risk engine (Notebooks 05 and 06) ----------
def line_matrix(line_pos, n_lines):
    """Sparse tower × line indicator used to combine tower failure probabilities into line failure probabilities."""
    n = len(line_pos)
    return csr_matrix((np.ones(n), (np.arange(n), line_pos)), shape=(n, n_lines))


def storm_probabilities(H, cell, gfac, theta_t, beta, M, scale=1.0, batch=512):
    """For every storm: expected damaged towers, line failure probabilities and per-tower annual sums.

    H: storms × cells storm-max 10-min wind; cell: grid cell of each tower; gfac: wind→IM factor per tower;
    theta_t: fragility median per tower (array); M: tower × line indicator."""
    n_s, n_l = H.shape[0], M.shape[1]
    E = np.zeros(n_s); P_line = np.zeros((n_s, n_l), np.float32); tower_sum = np.zeros(len(cell))
    lt = np.log(theta_t)
    for s in range(0, n_s, batch):
        im = H[s:s + batch][:, cell].astype(np.float32) * (gfac * scale)[None, :]
        p = norm.cdf((np.log(np.maximum(im, 1e-3)) - lt[None, :]) / beta)
        E[s:s + batch] = p.sum(1)
        tower_sum += p.sum(0)
        logq = (M.T @ np.log1p(-np.minimum(p, 1 - 1e-9)).T).T
        P_line[s:s + batch] = 1 - np.exp(logq)
    return E, P_line, tower_sum


def outage_mc(P_line, frm, to, n_sub, sub_pop, n_mc=30, min_p=1e-3, seed=7):
    """Expected population disconnected per storm (Raj et al.: population outside the largest connected component)."""
    rng = np.random.default_rng(seed)

    def lost(alive):
        g = csr_matrix((np.ones(alive.sum()), (frm[alive], to[alive])), shape=(n_sub, n_sub))
        nc, lab = connected_components(g, directed=False)
        big = np.bincount(lab).argmax()
        return sub_pop[lab != big].sum()
    base = lost(np.ones(len(frm), bool))
    out = np.zeros(P_line.shape[0])
    for s in np.where(P_line.max(1) > min_p)[0]:
        fails = rng.random((n_mc, P_line.shape[1])) < P_line[s][None, :]
        out[s] = np.mean([max(lost(~f) - base, 0.0) for f in fails])
    return out


def annual_from_events(values, sim_year, n_years):
    return np.bincount(np.asarray(sim_year, int), weights=np.asarray(values, float), minlength=n_years)


def rp_table(annual, rps=(10, 25, 50, 100, 250, 500)):
    return {T: float(np.quantile(annual, 1 - 1 / T)) for T in rps if T <= len(annual)}


def exceedance_curve(annual):
    a = np.sort(np.asarray(annual))[::-1]
    rank = np.arange(1, len(a) + 1)
    return pd.DataFrame({'annual_value': a, 'return_period_yr': len(a) / rank})
def build_risk_context(dirs, storm_10min_corr=1.0, kv_scope=None):
    """Load everything the risk engine needs: STORM hazard (03), towers with exposure (00/01), lines and population."""
    z = np.load(need(dirs['interim'] / 'hazard_storm_cells.npz', '03'), allow_pickle=False)
    th = pd.read_parquet(need(dirs['processed'] / 'towers_hazard.parquet', '03'))[['tower_id', 'cell']]
    towers = load_towers(dirs).merge(th, on='tower_id', how='inner')
    if kv_scope:
        towers = towers[towers.voltage_kv.isin(kv_scope)]
    lines, subs = load_network(dirs)
    lines = lines.sort_values('line_id').reset_index(drop=True)
    towers['line_pos'] = towers.line_id.map({lid: i for i, lid in enumerate(lines.line_id)})
    towers = towers.dropna(subset=['line_pos']).reset_index(drop=True)
    towers['line_pos'] = towers.line_pos.astype(int)
    n_sub = int(max(subs.sub_id.max(), lines.from_sub.max(), lines.to_sub.max()))
    sub_pop = np.zeros(n_sub)
    sub_pop[subs.sub_id.values.astype(int) - 1] = subs['pop'].values
    return dict(H=z['H'], years=z['sim_year'].astype(int), ids=z['storm_id'], n_years=int(z['n_years']),
                towers=towers, lines=lines, subs=subs, cell=towers.cell.values.astype(int),
                gfac=gust_factor(towers.exposure, 600) * storm_10min_corr, M=line_matrix(towers.line_pos.values, len(lines)),
                frm=lines.from_sub.values.astype(int) - 1, to=lines.to_sub.values.astype(int) - 1, n_sub=n_sub, sub_pop=sub_pop)


def risk_run(ctx, theta_t, beta, scale=1.0, n_mc=30, cost_cr=0.73, outage_days=7.0, kwh_pp_day=3.0, voll_inr=100.0, seed=7):
    """One full risk run over all STORM storms. Costs in INR crore. Returns events, annual totals and EAD."""
    theta_t = np.broadcast_to(np.asarray(theta_t, float), (len(ctx['cell']),))
    E, P_line, tsum = storm_probabilities(ctx['H'], ctx['cell'], ctx['gfac'], theta_t, beta, ctx['M'], scale)
    pop = outage_mc(P_line, ctx['frm'], ctx['to'], ctx['n_sub'], ctx['sub_pop'], n_mc=n_mc, seed=seed)
    ev = pd.DataFrame({'storm_id': ctx['ids'], 'sim_year': ctx['years'], 'exp_damaged_towers': E, 'exp_pop_disconnected': pop})
    ev['tower_cost_cr'] = E * cost_cr
    ev['ens_mwh'] = pop * outage_days * kwh_pp_day / 1000.0
    ev['outage_cost_cr'] = ev.ens_mwh * 1000.0 * voll_inr / 1e7
    ev['total_cost_cr'] = ev.tower_cost_cr + ev.outage_cost_cr
    cols = ['exp_damaged_towers', 'exp_pop_disconnected', 'tower_cost_cr', 'outage_cost_cr', 'total_cost_cr']
    annual = pd.DataFrame({c: annual_from_events(ev[c], ev.sim_year, ctx['n_years']) for c in cols})
    return dict(events=ev, annual=annual, P_line=P_line, tower_rate=tsum / ctx['n_years'],
                ead={c: float(annual[c].mean()) for c in cols})
# ==== end of shared functions ====

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box, MultiPoint
from shapely.ops import voronoi_diagram

ROOT, DIRS = setup_project()

# ---------------- settings ----------------
GEE_PROJECT = 'goggle-earth-499404'
MAX_COAST_KM = 100          # classify towers within this distance of the coast (hazard domain)
BUFFER_M = 500              # radius around each tower for roughness
BATCH = 1500                # towers per Earth Engine request
WORLDPOP_YEAR = 2020

# Typical roughness length z0 (m) for each ESA WorldCover class (engineering literature values; edit if needed)
WC_Z0 = {10: 0.80,   # tree cover
         20: 0.10,   # shrubland
         30: 0.03,   # grassland
         40: 0.05,   # cropland
         50: 0.70,   # built-up
         60: 0.005,  # bare / sparse vegetation
         70: 0.001,  # snow and ice
         80: 0.0002, # permanent water
         90: 0.03,   # herbaceous wetland
         95: 0.50,   # mangroves
         100: 0.01}  # moss and lichen

# z0 class limits (geometric means between neighbouring class values)
WMO_Z0 = {'off-sea': 0.013, 'off-land': 0.07, 'in-land': 0.18}            # TD-1555 Table 4.1
TC_Z0 = {1: 0.002, 2: 0.02, 3: 0.2, 4: 2.0}                                # IS 875 (Part 3):2015 terrain categories

towers = pd.read_parquet(need(DIRS['processed'] / 'towers.parquet', '00'))
gpkg = need(DIRS['processed'] / 'odisha_network.gpkg', '00')
subs = gpd.read_file(gpkg, layer='substations')
try:
    districts = gpd.read_file(gpkg, layer='districts')
except Exception:
    districts = None
dom = towers[towers.dist_coast_km <= MAX_COAST_KM].copy()
print(f'Towers within {MAX_COAST_KM} km of the coast: {len(dom):,} of {len(towers):,}')

## 1. Connect to Earth Engine
In Colab a sign-in window opens the first time. Use the Google account that owns the project.

In [ ]:
GEE_OK = False
try:
    import ee
    try:
        ee.Initialize(project=GEE_PROJECT)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT)
    print('Earth Engine ready:', ee.Number(1).getInfo() == 1)
    GEE_OK = True
except Exception as e:
    print('  ! Earth Engine not available:', type(e).__name__, str(e)[:300])
    print('    Check that the project is registered at https://code.earthengine.google.com/register '
          '(non-commercial) and the Earth Engine API is enabled. Continuing with fallbacks.')

## 2. Roughness around each tower (WorldCover → z₀)
We average ln(z₀) inside a 500 m circle (a geometric mean, which suits a log-law quantity) and also record the share of tree, built-up and water cover.

In [ ]:
def classify_wmo(z0):
    lim1 = math.sqrt(WMO_Z0['off-sea'] * WMO_Z0['off-land'])     # ≈ 0.030
    lim2 = math.sqrt(WMO_Z0['off-land'] * WMO_Z0['in-land'])     # ≈ 0.112
    return np.where(z0 <= lim1, 'off-sea', np.where(z0 <= lim2, 'off-land', 'in-land'))

def classify_tc(z0):
    lims = [math.sqrt(TC_Z0[1] * TC_Z0[2]), math.sqrt(TC_Z0[2] * TC_Z0[3]), math.sqrt(TC_Z0[3] * TC_Z0[4])]
    return 1 + np.searchsorted(lims, z0)

rough = None
if GEE_OK:
    wc = ee.ImageCollection('ESA/WorldCover/v200').first().select('Map')
    classes = list(WC_Z0.keys())
    lnz0 = wc.remap(classes, [math.log(WC_Z0[c]) for c in classes]).rename('ln_z0')
    img = (lnz0.addBands(wc.eq(10).Or(wc.eq(95)).rename('f_tree'))
               .addBands(wc.eq(50).rename('f_built')).addBands(wc.eq(80).rename('f_water')))
    rows = []
    for s in range(0, len(dom), BATCH):
        part = dom.iloc[s:s + BATCH]
        fc = ee.FeatureCollection([ee.Feature(ee.Geometry.Point([float(x), float(y)]), {'tower_id': int(i)})
                                   for i, x, y in zip(part.tower_id, part.lon, part.lat)])
        fc = fc.map(lambda f: f.buffer(BUFFER_M))
        res = img.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=20, tileScale=4).getInfo()
        rows += [f['properties'] for f in res['features']]
        print(f'  roughness: {min(s + BATCH, len(dom)):,}/{len(dom):,} towers')
    rough = pd.DataFrame(rows)
    rough['z0'] = np.exp(rough.ln_z0)
    rough['exposure_src'] = f'WorldCover v200, ln-mean z0 within {BUFFER_M} m'

if rough is None or rough.empty:
    rough = dom[['tower_id']].copy()
    rough['z0'] = np.where(dom.dist_coast_km.values <= 2.0, 0.01, 0.10)
    rough[['f_tree', 'f_built', 'f_water']] = np.nan
    rough['exposure_src'] = 'FALLBACK rule: ≤2 km from coast = off-sea, else in-land (Earth Engine not used)'

expo = rough[['tower_id', 'z0', 'f_tree', 'f_built', 'f_water', 'exposure_src']].copy()
expo['exposure'] = classify_wmo(expo.z0.values)
if 'FALLBACK' in expo.exposure_src.iloc[0]:
    expo['exposure'] = np.where(expo.z0 <= 0.03, 'off-sea', 'in-land')
expo['terrain_cat'] = classify_tc(expo.z0.values)
if 'FALLBACK' in expo.exposure_src.iloc[0]:
    expo['terrain_cat'] = 2          # unknown without land cover: use the code's reference category
expo = expo.merge(dom[['tower_id', 'dist_coast_km', 'band', 'voltage_kv', 'lon', 'lat']], on='tower_id')
expo.to_parquet(DIRS['processed'] / 'towers_exposure.parquet', index=False)

print('\nExposure class by coastal band (share of towers):')
print(pd.crosstab(expo.band, expo.exposure, normalize='index').round(2).to_string())
print('\nIS 875/802 terrain category by band (share):')
print(pd.crosstab(expo.band, expo.terrain_cat, normalize='index').round(2).to_string())
print('\nGust factors that follow (3-s gust / 10-min mean):',
      {k: GUST3[k][600] for k in ['off-sea', 'off-land', 'in-land']})

## 3. Population served by each substation
Service area = Voronoi polygon around each Odisha substation, clipped to the state boundary. WorldPop 2020 (100 m) is summed inside each polygon. This replaces Raj et al.'s district-based weights where it works, and we compare the two.

In [ ]:
if districts is not None and len(districts):
    state = union(districts.to_crs(CRS_GEO))
else:
    state = box(81.3, 17.8, 87.6, 22.6)
in_state = subs[subs.within(state.buffer(0.05))].copy()
vor = voronoi_diagram(MultiPoint(list(in_state.geometry)), envelope=state.envelope.buffer(1.0))
cells = gpd.GeoDataFrame(geometry=list(vor.geoms), crs=CRS_GEO)
cells = gpd.sjoin(cells, in_state[['sub_id', 'name', 'geometry']], predicate='contains', how='inner').drop(columns='index_right')
cells['geometry'] = cells.geometry.intersection(state)
cells = cells[~cells.geometry.is_empty]
print(f'Service areas: {len(cells)} substations inside Odisha (of {len(subs)} in the network file)')

pop = None
if GEE_OK:
    wp = (ee.ImageCollection('WorldPop/GP/100m/pop').filter(ee.Filter.eq('country', 'IND'))
            .filter(ee.Filter.eq('year', WORLDPOP_YEAR)).mosaic().select('population'))
    feats = []
    for sid, g in zip(cells.sub_id, cells.geometry.simplify(0.003)):
        gj = g.__geo_interface__
        feats.append(ee.Feature(ee.Geometry(gj), {'sub_id': int(sid)}))
    rows = []
    for s in range(0, len(feats), 60):
        res = wp.reduceRegions(collection=ee.FeatureCollection(feats[s:s + 60]), reducer=ee.Reducer.sum(),
                               scale=100, tileScale=8).getInfo()
        rows += [f['properties'] for f in res['features']]
    pop = pd.DataFrame(rows).rename(columns={'sum': 'pop_worldpop'})
    print(f'WorldPop {WORLDPOP_YEAR} total in service areas: {pop.pop_worldpop.sum() / 1e6:.1f} million '
          '(Odisha ≈ 42 million in Census 2011; WorldPop 2020 should be somewhat higher)')
else:
    print('  ! WorldPop step skipped (no Earth Engine). Later notebooks will use Raj et al. population weights.')

sp = in_state[['sub_id', 'name', 'pop_weight', 'dist_coast_km']].copy()
if pop is not None:
    sp = sp.merge(pop[['sub_id', 'pop_worldpop']], on='sub_id', how='left')
    sp.to_csv(DIRS['processed'] / 'substations_population.csv', index=False)
    r = np.corrcoef(sp.pop_weight.fillna(0), sp.pop_worldpop.fillna(0))[0, 1]
    print(f'Correlation between WorldPop service-area population and Raj et al. weights: r = {r:.2f}')
    print(sp.sort_values('pop_worldpop', ascending=False).head(10).to_string(index=False))
cells.merge(sp, on=['sub_id', 'name'], how='left').to_file(DIRS['processed'] / 'service_areas.gpkg', driver='GPKG')

## 4. QA map

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 8))
colors = {'off-sea': '#2166ac', 'off-land': '#92c5de', 'in-land': '#b2182b'}
for k, c in colors.items():
    d = expo[expo.exposure == k]
    ax[0].scatter(d.lon, d.lat, s=1, c=c, label=f'{k} ({len(d):,})')
ax[0].set_title('WMO exposure class of each tower (sets the gust factor)')
ax[0].legend(markerscale=8, fontsize=8); ax[0].set_aspect('equal')
cm = cells.merge(sp, on=['sub_id', 'name'], how='left')
col = 'pop_worldpop' if 'pop_worldpop' in cm else 'pop_weight'
cm.plot(column=col, ax=ax[1], cmap='YlOrRd', legend=True, edgecolor='grey', linewidth=0.2,
        legend_kwds={'label': f'population served ({col})', 'shrink': 0.6})
ax[1].set_title('Substation service areas and population served'); ax[1].set_aspect('equal')
plt.tight_layout(); fig.savefig(DIRS['figures'] / '01_exposure_population.png', dpi=200); plt.show()
print('Saved towers_exposure.parquet, substations_population.csv (if WorldPop ran), service_areas.gpkg')